# AgentCore Gateway - 헤더 및 쿼리 파라미터 전파

## 개요

이 튜토리얼에서는 클라이언트의 사용자 지정 HTTP 헤더와 쿼리 파라미터를 대상으로 전파하는 AgentCore Gateway의 기능을 살펴봅니다. 이 기능을 사용하면 사용자 지정 인터셉터 코드 없이도 분산 추적, 멀티 테넌트 격리, API 버전 관리, 속도 제한과 같은 엔터프라이즈 패턴을 구현할 수 있습니다.

### 헤더 및 쿼리 파라미터 전파가 중요한 이유

최신 엔터프라이즈 애플리케이션에서는 API 호출을 통해 컨텍스트 정보를 전달해야 합니다.

- **분산 추적**: 상관관계 ID로 마이크로서비스 전반의 요청을 추적합니다.
- **멀티 테넌시**: 테넌트 식별자로 데이터를 적절히 격리합니다.
- **API 버전 관리**: 버전 파라미터를 사용하여 적절한 구현으로 라우팅합니다.
- **환경 라우팅**: 환경 플래그로 스테이징 환경과 프로덕션 환경의 동작을 제어합니다.
- **속도 제한**: 응답 헤더로 할당량 정보를 전달합니다.

### 헤더 전파 방식과 인터셉터 방식 비교

AgentCore Gateway는 헤더 처리를 위해 다음 두 가지 방식을 제공합니다.

1. **헤더 전파**(이 튜토리얼): 특정 헤더와 쿼리 파라미터를 자동으로 전달하도록 `metadataConfiguration`을 구성합니다. 상관관계 ID, 테넌트 ID, API 버전과 같은 사용자 지정 헤더에 적합합니다.

2. **인터셉터 Lambda**(튜토리얼 14-token-exchange-at-request-interceptor): Authorization 헤더 토큰 교환, 사용자 지정 인증 로직, 동적 헤더 변환과 같이 보안에 민감한 시나리오에서는 인터셉터 Lambda를 사용합니다.

### 헤더 전파 작동 방식

대상을 생성할 때 전파할 헤더와 쿼리 파라미터를 지정합니다.

```python
"metadataConfiguration": {
    "allowedRequestHeaders": ["x-correlation-id", "x-tenant-id"],
    "allowedResponseHeaders": ["x-rate-limit-remaining"],
    "allowedQueryParameters": ["version", "environment"]
}
```

Gateway는 다음 작업을 자동으로 수행합니다.
1. 클라이언트 요청에서 지정된 헤더와 쿼리 파라미터를 추출합니다.
2. 적절한 이벤트 구조로 대상 Lambda에 전달합니다.
3. 지정된 응답 헤더를 클라이언트에 반환합니다.

### 튜토리얼 세부 정보

| 정보                 | 세부 정보                                                 |
|:---------------------|:----------------------------------------------------------|
| 튜토리얼 유형        | 대화형                                                     |
| AgentCore 구성 요소  | AgentCore Gateway                                         |
| Gateway 대상 유형    | AWS Lambda                                                |
| 인바운드 인증        | OAuth (Cognito)                                           |
| 아웃바운드 인증      | AWS IAM                                                   |
| 튜토리얼 구성        | AgentCore Gateway 생성 및 호출                            |
| 튜토리얼 분야        | 전 분야                                                    |
| 예제 난이도          | 쉬움                                                       |
| 사용 SDK             | boto3                                                     |

### 튜토리얼 아키텍처

![아키텍처 다이어그램](images/08-custom-header-propagation.png)

---

**아키텍처 흐름:**

1. **클라이언트**: 사용자 지정 헤더(x-correlation-id, x-tenant-id)와 쿼리 파라미터(version, environment)를 포함한 요청을 전송합니다.
2. **AgentCore Gateway**: 특정 헤더와 쿼리 파라미터를 허용하도록 metadataConfiguration을 구성합니다.
3. **대상 Lambda(MCP Server)**: 적절한 이벤트 구조로 헤더와 쿼리 파라미터를 수신하고 응답 헤더를 반환합니다.

## 1단계: 종속성 설치

이 단계에서는 MCP Gateway를 생성하고 AWS 서비스와 상호 작용하는 데 필요한 패키지를 설치합니다.

In [ ]:
import subprocess
import sys

# 필수 패키지 설치
subprocess.check_call([sys.executable, "-m", "pip", "install", "boto3", "requests"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "strands-agents"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "setuptools", "pip"])

print("Dependencies installed successfully")

## 2단계: 설정 및 구성

필수 라이브러리를 가져오고 AWS 클라이언트를 초기화합니다.

In [ ]:
import boto3
import json
import time
import zipfile
import io
from datetime import datetime
from botocore.exceptions import ClientError

# 고유한 리소스 이름을 위한 타임스탬프 초기화
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")

# 리전 설정
region = "us-east-1"

# AWS 클라이언트 초기화
lambda_client = boto3.client("lambda", region_name=region)
iam_client = boto3.client("iam", region_name=region)
agentcore_client = boto3.client("bedrock-agentcore-control", region_name=region)
cognito_client = boto3.client("cognito-idp", region_name=region)
sts_client = boto3.client("sts", region_name=region)

print(f"AWS clients initialized for region: {region}")
print(f"Timestamp for resource naming: {timestamp}")

## 3단계: OAuth 인증용 Cognito User Pool 생성

이 단계에서는 OAuth 인증을 위한 독립적인 Cognito 환경을 구성합니다. 따라서 수동으로 Cognito를 구성하지 않아도 Notebook을 완전히 자동화할 수 있습니다.

**생성되는 항목:**
- 인증용 Cognito User Pool
- OAuth 범위(read, write)가 있는 Resource Server
- 클라이언트 자격 증명 흐름용 App Client
- OAuth 엔드포인트용 User Pool Domain

**내보내는 변수:**
- `discovery_url` - Gateway 구성에 사용
- `client_id` - Gateway 구성 및 테스트에 사용
- `client_secret` - 테스트에 사용
- `token_endpoint` - 테스트에 사용

In [ ]:
# Cognito User Pool 생성
user_pool_name = f"header-propagation-pool-{timestamp}"
user_pool_response = cognito_client.create_user_pool(
    PoolName=user_pool_name,
    Policies={
        "PasswordPolicy": {
            "MinimumLength": 8,
            "RequireUppercase": False,
            "RequireLowercase": False,
            "RequireNumbers": False,
            "RequireSymbols": False,
        }
    },
)
user_pool_id = user_pool_response["UserPool"]["Id"]
print(f"User Pool created: {user_pool_id}")

# Resource Server 생성
resource_server_identifier = f"header-propagation-api-{timestamp}"
cognito_client.create_resource_server(
    UserPoolId=user_pool_id,
    Identifier=resource_server_identifier,
    Name=f"HeaderPropagationAPI-{timestamp}",
    Scopes=[
        {"ScopeName": "read", "ScopeDescription": "Read access"},
        {"ScopeName": "write", "ScopeDescription": "Write access"},
    ],
)
print(f"Resource Server created: {resource_server_identifier}")

# App Client 생성
app_client_response = cognito_client.create_user_pool_client(
    UserPoolId=user_pool_id,
    ClientName=f"header-propagation-client-{timestamp}",
    GenerateSecret=True,
    AllowedOAuthFlows=["client_credentials"],
    AllowedOAuthScopes=[
        f"{resource_server_identifier}/read",
        f"{resource_server_identifier}/write",
    ],
    AllowedOAuthFlowsUserPoolClient=True,
)
client_id = app_client_response["UserPoolClient"]["ClientId"]
client_secret = app_client_response["UserPoolClient"]["ClientSecret"]
print(f"App Client created: {client_id}")

# User Pool Domain 생성
domain_name = f"header-prop-{timestamp}"
cognito_client.create_user_pool_domain(Domain=domain_name, UserPoolId=user_pool_id)
print(f"Domain created: {domain_name}")

# OAuth 엔드포인트 구성
discovery_url = f"https://cognito-idp.{region}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration"
token_endpoint = f"https://{domain_name}.auth.{region}.amazoncognito.com/oauth2/token"

print("\nOAuth Configuration:")
print(f"  Discovery URL: {discovery_url}")
print(f"  Token Endpoint: {token_endpoint}")
print(f"  Client ID: {client_id}")
print(f"  Client Secret: {client_secret[:20]}...")

## 4단계: OAuth 인증을 사용하는 AgentCore Gateway 생성

이 단계에서는 2.5단계에서 생성한 Cognito User Pool을 사용하여 OAuth 인증이 구성된 AgentCore Gateway를 생성합니다.

**핵심 사항:**
- Cognito를 통한 OAuth 인증
- 헤더와 쿼리 파라미터는 대상 수준에서 구성(5단계)
- Gateway는 대상의 metadataConfiguration에 따라 구성된 헤더를 자동으로 전달

In [ ]:
# Gateway용 IAM 역할 생성
gateway_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

iam_response = iam_client.create_role(
    RoleName=f"BedrockAgentCoreGatewayRole-{timestamp}",
    AssumeRolePolicyDocument=json.dumps(gateway_trust_policy),
    Description="IAM role for AgentCore Gateway",
)

# Lambda 호출 정책 연결
iam_client.put_role_policy(
    RoleName=f"BedrockAgentCoreGatewayRole-{timestamp}",
    PolicyName="LambdaInvokePolicy",
    PolicyDocument=json.dumps(
        {
            "Version": "2012-10-17",
            "Statement": [
                {
                    "Effect": "Allow",
                    "Action": ["lambda:InvokeFunction"],
                    "Resource": "*",
                }
            ],
        }
    ),
)

# 역할을 사용할 수 있을 때까지 대기
print("Waiting for IAM role to be available...")
time.sleep(10)

# Gateway 생성
gateway_response = agentcore_client.create_gateway(
    name=f"header-propagation-gateway-{timestamp}",
    protocolType="MCP",
    protocolConfiguration={"mcp": {"supportedVersions": ["2025-03-26", "2025-06-18"]}},
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedClients": [client_id],
        }
    },
    roleArn=iam_response["Role"]["Arn"],
)

gateway_id = gateway_response["gatewayId"]

# Gateway 생성 여부 확인
print(f"Gateway created with status: {gateway_response['status']}")
print(f"Gateway ID: {gateway_id}")

# Gateway가 준비될 때까지 대기
print("\nWaiting for gateway to be ready...")
max_wait = 300
wait_interval = 10
elapsed = 0

while elapsed < max_wait:
    gateway_status = agentcore_client.get_gateway(gatewayIdentifier=gateway_id)
    status = gateway_status["status"]
    print(f"Gateway status: {status}")

    if status == "READY":
        gateway_url = gateway_status["gatewayUrl"]
        print("\nGateway is ready!")
        print(f"Gateway URL: {gateway_url}")
        break
    elif status in ["FAILED", "DELETING", "DELETED"]:
        print(f"Gateway creation failed with status: {status}")
        break

    time.sleep(wait_interval)
    elapsed += wait_interval

if elapsed >= max_wait:
    print("Gateway creation timed out")

## 5단계: metadataConfiguration을 사용하는 대상 Lambda 생성

이 단계에서는 Gateway의 도구 호출을 수신할 대상 Lambda 함수를 생성합니다. 핵심 기능은 전파할 헤더와 쿼리 파라미터를 지정하는 `metadataConfiguration`입니다.

**metadataConfiguration 필드:**
- `allowedRequestHeaders`: Lambda에 전달할 요청 헤더 목록(예: x-correlation-id, x-tenant-id)
- `allowedResponseHeaders`: 클라이언트에 반환할 응답 헤더 목록(주로 MCP Server 대상에 사용)
- `allowedQueryParameters`: Lambda에 전달할 쿼리 파라미터 목록(예: version, environment)

**MCP Lambda 대상에서 작동하는 방식:**
1. 클라이언트가 사용자 지정 헤더와 쿼리 파라미터를 포함한 MCP 요청을 전송합니다.
2. Gateway가 허용된 헤더와 파라미터만 추출합니다.
3. Gateway가 MCP 프로토콜을 해제하고 다음 정보를 포함하여 Lambda를 호출합니다.
   - 이벤트에 직접 포함된 도구 인수(예: `event['message']`)
   - `context.client_context.custom['bedrockAgentCorePropagatedHeaders']`의 헤더
   - `context.client_context.custom['bedrockAgentCorePropagatedQueryParameters']`의 쿼리 파라미터
4. Lambda가 요청을 처리하고 응답을 반환합니다.
5. Gateway가 응답을 MCP 형식으로 래핑하여 클라이언트에 전달합니다.

**보안:** 명시적으로 허용된 헤더와 파라미터만 전달됩니다. 이를 통해 민감한 데이터가 실수로 노출되는 것을 방지합니다.

In [ ]:
# 대상 Lambda 함수 코드 정의
target_lambda_code = '''
import json
import logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)

def lambda_handler(event, context):
    """Header 및 Query Parameter Propagation을 적용해 도구 호출을 처리합니다.
    
    MCP Lambda target에서 gateway는 다음 작업을 수행합니다.
    - MCP 프로토콜을 해제하고 도구 인수를 event에 직접 전달합니다.
    - 전달된 header를 context.client_context.custom['bedrockAgentCorePropagatedHeaders']에 넣습니다.
    - 전달된 query parameter를 context.client_context.custom['bedrockAgentCorePropagatedQueryParameters']에 넣습니다.
    """
    
    logger.info(f"Received event: {json.dumps(event)}")
    
    # Context에서 전달된 header와 query parameter 추출
    correlation_id = 'not-provided'
    tenant_id = 'not-provided'
    version = 'not-provided'
    environment_param = 'not-provided'
    
    if hasattr(context, 'client_context') and context.client_context:
        custom = context.client_context.custom
        
        # 전달된 header 추출
        propagated_headers = custom.get('bedrockAgentCorePropagatedHeaders', {})
        correlation_id = propagated_headers.get('x-correlation-id', correlation_id)
        tenant_id = propagated_headers.get('x-tenant-id', tenant_id)
        
        # 전달된 query parameter 추출
        propagated_query_params = custom.get('bedrockAgentCorePropagatedQueryParameters', {})
        version = propagated_query_params.get('version', version)
        environment_param = propagated_query_params.get('environment', environment_param)
    
    logger.info(f"Propagated Headers - Correlation ID: {correlation_id}, Tenant ID: {tenant_id}")
    logger.info(f"Propagated Query Params - Version: {version}, Environment: {environment_param}")
    
    # Tool argument 추출(Gateway가 직접 전송)
    message = event.get('message', 'No message provided')
    
    # 사용자 지정 header가 포함된 응답 반환
    # gateway가 추출할 수 있도록 최상위 수준에서 header 반환 시도
    return {
        "message": message,
        "propagated_headers": {
            "x-correlation-id": correlation_id,
            "x-tenant-id": tenant_id
        },
        "propagated_query_params": {
            "version": version,
            "environment": environment_param
        },
        "note": "Headers and query parameters successfully propagated via metadataConfiguration",
        "headers": {
            "x-rate-limit-remaining": "95"
        }
    }
'''


# Lambda용 ZIP 파일 생성
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zip_file:
    zip_file.writestr("lambda_function.py", target_lambda_code)
zip_buffer.seek(0)

# 대상 Lambda용 IAM 역할 생성
target_lambda_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

target_lambda_role_response = iam_client.create_role(
    RoleName=f"TargetLambdaRole-{timestamp}",
    AssumeRolePolicyDocument=json.dumps(target_lambda_trust_policy),
    Description="IAM role for Target Lambda",
)

# 기본 Lambda 실행 정책 연결
iam_client.attach_role_policy(
    RoleName=f"TargetLambdaRole-{timestamp}",
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
)

print("Waiting for Lambda role to be available...")
time.sleep(10)

# 대상 Lambda 함수 생성
target_lambda_response = lambda_client.create_function(
    FunctionName=f"header-propagation-target-{timestamp}",
    Runtime="python3.12",
    Role=target_lambda_role_response["Role"]["Arn"],
    Handler="lambda_function.lambda_handler",
    Code={"ZipFile": zip_buffer.read()},
    Description="Target Lambda with Header and Query Parameter Propagation",
    Timeout=30,
    MemorySize=256,
)

target_lambda_arn = target_lambda_response["FunctionArn"]
print(f"Target Lambda created: {target_lambda_arn}")

# Lambda가 준비될 때까지 대기
print("Waiting for Lambda function to be ready...")
time.sleep(15)

# Lambda 활성 상태 확인
max_wait = 60
wait_interval = 5
elapsed = 0

while elapsed < max_wait:
    try:
        lambda_status = lambda_client.get_function(FunctionName=f"header-propagation-target-{timestamp}")
        state = lambda_status["Configuration"]["State"]
        print(f"Lambda state: {state}")
        if state == "Active":
            print("Lambda is ready!")
            break
    except Exception as e:
        print(f"Checking Lambda status: {e}")

    time.sleep(wait_interval)
    elapsed += wait_interval

# metadataConfiguration을 사용하는 Gateway 대상 생성
print("\nCreating gateway target with metadataConfiguration...")
target_response = agentcore_client.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name=f"header-propagation-target-{timestamp}",
    targetConfiguration={
        "mcp": {
            "lambda": {
                "lambdaArn": target_lambda_arn,
                "toolSchema": {
                    "inlinePayload": [
                        {
                            "name": "echo",
                            "description": "Echoes back the input with context information including propagated headers and query parameters",
                            "inputSchema": {
                                "type": "object",
                                "properties": {
                                    "message": {
                                        "type": "string",
                                        "description": "Message to echo",
                                    }
                                },
                                "required": ["message"],
                            },
                        }
                    ]
                },
            }
        }
    },
    # 핵심 기능: 헤더/쿼리 파라미터 전파를 위한 metadataConfiguration
    metadataConfiguration={
        "allowedRequestHeaders": ["x-correlation-id", "x-tenant-id"],
        "allowedResponseHeaders": ["x-rate-limit-remaining"],
        "allowedQueryParameters": ["version", "environment"],
    },
    credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}],
)

target_id = target_response["targetId"]
print(f"Gateway target created: {target_id}")
print("\nmetadataConfiguration:")
print(f"  Allowed Request Headers: {target_response['metadataConfiguration']['allowedRequestHeaders']}")
print(f"  Allowed Response Headers: {target_response['metadataConfiguration']['allowedResponseHeaders']}")
print(f"  Allowed Query Parameters: {target_response['metadataConfiguration']['allowedQueryParameters']}")

## 6단계: 요약 및 리소스 정보

이 단계에서는 생성된 모든 리소스의 요약 정보를 표시합니다.

In [ ]:
print("=" * 80)
print("RESOURCE SUMMARY")
print("=" * 80)
print("\nCognito Resources:")
print(f"  User Pool ID: {user_pool_id}")
print(f"  Client ID: {client_id}")
print(f"  Discovery URL: {discovery_url}")
print(f"  Token Endpoint: {token_endpoint}")
print("\nGateway Resources:")
print(f"  Gateway ID: {gateway_id}")
print(f"  Gateway URL: {gateway_url}")
print(f"  Target ID: {target_id}")
print(f"  Target Lambda ARN: {target_lambda_arn}")
print("\nmetadataConfiguration:")
print("  Request Headers: x-correlation-id, x-tenant-id")
print("  Response Headers: x-rate-limit-remaining")
print("  Query Parameters: version, environment")
print("\n" + "=" * 80)

## 7단계: 헤더 및 쿼리 파라미터 전파 테스트

이 단계에서는 다음과 같이 헤더 전파 기능을 테스트합니다.
1. Cognito에서 OAuth 토큰을 가져옵니다.
2. 사용자 지정 헤더(x-correlation-id, x-tenant-id)를 포함한 요청을 전송합니다.
3. URL에 쿼리 파라미터(version, environment)를 추가합니다.
4. Lambda가 헤더와 쿼리 파라미터를 수신하는지 확인합니다.
5. 응답 헤더가 클라이언트에 반환되는지 확인합니다.

In [ ]:
import requests
import base64
import json

# Cognito에서 OAuth 토큰 가져오기
auth_string = f"{client_id}:{client_secret}"
auth_bytes = auth_string.encode("utf-8")
auth_b64 = base64.b64encode(auth_bytes).decode("utf-8")

token_response = requests.post(
    token_endpoint,
    headers={
        "Content-Type": "application/x-www-form-urlencoded",
        "Authorization": f"Basic {auth_b64}",
    },
    data={
        "grant_type": "client_credentials",
        "scope": f"{resource_server_identifier}/read {resource_server_identifier}/write",
    },
)

access_token = token_response.json()["access_token"]

# 테스트 데이터 준비
correlation_id = "trace-a1b2c3d4"
tenant_id = "tenant-acme-corp"
version = "v1"
environment = "prod"
test_url = f"{gateway_url}?version={version}&environment={environment}"

# 도구 이름 가져오기
tools_response = requests.post(
    test_url,
    headers={
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
        "x-correlation-id": correlation_id,
        "x-tenant-id": tenant_id,
    },
    json={"jsonrpc": "2.0", "id": 1, "method": "tools/list"},
)
tool_name = tools_response.json()["result"]["tools"][0]["name"]

# 도구 호출
response = requests.post(
    test_url,
    headers={
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
        "x-correlation-id": correlation_id,
        "x-tenant-id": tenant_id,
    },
    json={
        "jsonrpc": "2.0",
        "id": 2,
        "method": "tools/call",
        "params": {
            "name": tool_name,
            "arguments": {"message": "Testing Header and Query Parameter Propagation"},
        },
    },
)

# 응답 출력
print("RESPONSE HEADERS:")
for header, value in response.headers.items():
    print(f"  {header}: {value}")

# 사용자 지정 응답 헤더 확인
if "x-rate-limit-remaining" in response.headers:
    print(
        f"\n  >>> CUSTOM RESPONSE HEADER FOUND: x-rate-limit-remaining = {response.headers['x-rate-limit-remaining']}"
    )
else:
    print("\n  >>> Custom response header 'x-rate-limit-remaining' not found in response")

print("\nRESPONSE BODY:")
print(json.dumps(response.json(), indent=2))

## 8단계: 리소스 정리

이 단계에서는 불필요한 AWS 비용이 발생하지 않도록 이 튜토리얼에서 생성한 모든 리소스를 삭제합니다.

삭제할 리소스:
- Gateway 대상
- Gateway
- 대상 Lambda 함수
- 대상 Lambda IAM 역할
- Gateway IAM 역할
- Cognito 사용자 풀 도메인
- Cognito 앱 클라이언트
- Cognito 리소스 서버
- Cognito 사용자 풀

**주의:** 이 작업은 되돌릴 수 없습니다. 계속하기 전에 이러한 리소스를 삭제할지 확인하세요.

In [ ]:
print("Starting cleanup process...\n")

# 1. Gateway 대상 삭제
print("1. Deleting Gateway Target...")
try:
    if "target_id" in locals() and "gateway_id" in locals():
        agentcore_client.delete_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)
        print(f"   Deleted Gateway Target: {target_id}")

        # 대상 삭제가 완료될 때까지 대기
        print("   Waiting for target deletion to complete...")
        time.sleep(10)
    else:
        print("   Gateway Target not found in session")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceNotFoundException":
        print("   Gateway Target already deleted")
    else:
        print(f"   Error deleting Gateway Target: {e}")

# 2. Gateway 삭제
print("\n2. Deleting Gateway...")
try:
    if "gateway_id" in locals():
        agentcore_client.delete_gateway(gatewayIdentifier=gateway_id)
        print(f"   Deleted Gateway: {gateway_id}")

        # Gateway 삭제가 완료될 때까지 대기
        print("   Waiting for gateway deletion to complete...")
        time.sleep(10)
    else:
        print("   Gateway not found in session")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceNotFoundException":
        print("   Gateway already deleted")
    else:
        print(f"   Error deleting Gateway: {e}")

# 3. 대상 Lambda 함수 삭제
print("\n3. Deleting Target Lambda Function...")
try:
    if "target_lambda_arn" in locals():
        lambda_client.delete_function(FunctionName=f"header-propagation-target-{timestamp}")
        print(f"   Deleted Lambda Function: header-propagation-target-{timestamp}")
    else:
        print("   Lambda Function not found in session")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceNotFoundException":
        print("   Lambda Function already deleted")
    else:
        print(f"   Error deleting Lambda Function: {e}")

# 4. 대상 Lambda IAM 역할 삭제
print("\n4. Deleting Target Lambda IAM Role...")
try:
    # 먼저 관리형 정책 분리
    iam_client.detach_role_policy(
        RoleName=f"TargetLambdaRole-{timestamp}",
        PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
    )
    print("   Detached managed policy from Target Lambda Role")

    # 역할 삭제
    iam_client.delete_role(RoleName=f"TargetLambdaRole-{timestamp}")
    print(f"   Deleted IAM Role: TargetLambdaRole-{timestamp}")
except ClientError as e:
    if e.response["Error"]["Code"] == "NoSuchEntity":
        print("   Target Lambda IAM Role already deleted")
    else:
        print(f"   Error deleting Target Lambda IAM Role: {e}")

# 5. Gateway IAM 역할 삭제
print("\n5. Deleting Gateway IAM Role...")
try:
    # 먼저 인라인 정책 삭제
    iam_client.delete_role_policy(
        RoleName=f"BedrockAgentCoreGatewayRole-{timestamp}",
        PolicyName="LambdaInvokePolicy",
    )
    print("   Deleted inline policy from Gateway Role")

    # 역할 삭제
    iam_client.delete_role(RoleName=f"BedrockAgentCoreGatewayRole-{timestamp}")
    print(f"   Deleted IAM Role: BedrockAgentCoreGatewayRole-{timestamp}")
except ClientError as e:
    if e.response["Error"]["Code"] == "NoSuchEntity":
        print("   Gateway IAM Role already deleted")
    else:
        print(f"   Error deleting Gateway IAM Role: {e}")

# 6. Cognito User Pool Domain 삭제
print("\n6. Deleting Cognito User Pool Domain...")
try:
    if "domain_name" in locals() and "user_pool_id" in locals():
        cognito_client.delete_user_pool_domain(Domain=domain_name, UserPoolId=user_pool_id)
        print(f"   Deleted User Pool Domain: {domain_name}")

        # 도메인 삭제가 완료될 때까지 대기
        print("   Waiting for domain deletion to complete...")
        time.sleep(10)
    else:
        print("   User Pool Domain not found in session")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceNotFoundException":
        print("   User Pool Domain already deleted")
    else:
        print(f"   Error deleting User Pool Domain: {e}")

# 7. Cognito App Client 삭제
print("\n7. Deleting Cognito App Client...")
try:
    if "client_id" in locals() and "user_pool_id" in locals():
        cognito_client.delete_user_pool_client(UserPoolId=user_pool_id, ClientId=client_id)
        print(f"   Deleted App Client: {client_id}")
    else:
        print("   App Client not found in session")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceNotFoundException":
        print("   App Client already deleted")
    else:
        print(f"   Error deleting App Client: {e}")

# 8. Cognito Resource Server 삭제
print("\n8. Deleting Cognito Resource Server...")
try:
    if "resource_server_identifier" in locals() and "user_pool_id" in locals():
        cognito_client.delete_resource_server(UserPoolId=user_pool_id, Identifier=resource_server_identifier)
        print(f"   Deleted Resource Server: {resource_server_identifier}")
    else:
        print("   Resource Server not found in session")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceNotFoundException":
        print("   Resource Server already deleted")
    else:
        print(f"   Error deleting Resource Server: {e}")

# 9. Cognito User Pool 삭제
print("\n9. Deleting Cognito User Pool...")
try:
    if "user_pool_id" in locals():
        cognito_client.delete_user_pool(UserPoolId=user_pool_id)
        print(f"   Deleted User Pool: {user_pool_id}")
    else:
        print("   User Pool not found in session")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceNotFoundException":
        print("   User Pool already deleted")
    else:
        print(f"   Error deleting User Pool: {e}")

print("\nCleanup completed!")